# Chuyển đổi dữ liệu từ CSV sang Parquet

Do file `train.csv` có dung lượng lên đến 4.7GB, việc đọc bằng `pd.read_csv` thông thường sẽ gây tràn RAM (máy chỉ còn khoảng 2-3GB RAM trống).
Do đó chúng ta sẽ:
1. Đọc file `train.csv` theo từng chunk (phần nhỏ).
2. Ép kiểu dữ liệu (downcast) để tối ưu dung lượng bộ nhớ.
3. Ghi nối tiếp vào một file `train.parquet` duy nhất sử dụng `pyarrow.parquet.ParquetWriter`. Parquet nén tốt và truy xuất siêu nhanh.

In [1]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# Định nghĩa các thư mục
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# Tạo thư mục processed nếu chưa có
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [2]:
# Đường dẫn file
train_csv = DATA_RAW / "train.csv"
train_parquet = DATA_PROCESSED / "train.parquet"

print(f"Nguồn: {train_csv}")
print(f"Đích: {train_parquet}")

Nguồn: /home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast/data/raw/train.csv
Đích: /home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast/data/processed/train.parquet


In [3]:
# Đọc và ghi chunksize
chunksize = 1_000_000
writer = None

# Ép kiểu dữ liệu để tối ưu dung lượng RAM
dtypes = {
    "id": "int32",
    "store_nbr": "int16",
    "item_nbr": "int32",
    "unit_sales": "float32",
    # onpromotion có giá trị NaN nên để là object trước, sau đó ép kiểu nullable boolean
}

for i, chunk in enumerate(pd.read_csv(
    train_csv,
    chunksize=chunksize,
    parse_dates=["date"],
    dtype=dtypes
)):
    # Ép kiểu nullable boolean (hỗ trợ True, False, và <NA>)
    chunk["onpromotion"] = chunk["onpromotion"].astype("boolean")
    
    # Chuyển sang bảng pyarrow
    table = pa.Table.from_pandas(chunk)
    
    # Khởi tạo writer ở chunk đầu tiên để lấy đúng schema
    if writer is None:
        writer = pq.ParquetWriter(train_parquet, table.schema)
    
    # Ghi table
    writer.write_table(table)
    
    print(f"Processed chunk {i+1} | rows={len(chunk):,} | last_date={chunk['date'].max().date()}")

if writer:
    writer.close()
    
print("Hoàn tất chuyển đổi CSV sang Parquet!")

Processed chunk 1 | rows=1,000,000 | last_date=2013-01-26
Processed chunk 2 | rows=1,000,000 | last_date=2013-02-19
Processed chunk 3 | rows=1,000,000 | last_date=2013-03-15
Processed chunk 4 | rows=1,000,000 | last_date=2013-04-08
Processed chunk 5 | rows=1,000,000 | last_date=2013-05-01
Processed chunk 6 | rows=1,000,000 | last_date=2013-05-24
Processed chunk 7 | rows=1,000,000 | last_date=2013-06-15
Processed chunk 8 | rows=1,000,000 | last_date=2013-07-07
Processed chunk 9 | rows=1,000,000 | last_date=2013-07-30
Processed chunk 10 | rows=1,000,000 | last_date=2013-08-21
Processed chunk 11 | rows=1,000,000 | last_date=2013-09-11
Processed chunk 12 | rows=1,000,000 | last_date=2013-10-03
Processed chunk 13 | rows=1,000,000 | last_date=2013-10-25
Processed chunk 14 | rows=1,000,000 | last_date=2013-11-15
Processed chunk 15 | rows=1,000,000 | last_date=2013-12-05
Processed chunk 16 | rows=1,000,000 | last_date=2013-12-24
Processed chunk 17 | rows=1,000,000 | last_date=2014-01-12
Proces

/tmp/ipykernel_107272/2494204672.py:14: DtypeWarning: Columns (0: onpromotion) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(


Processed chunk 22 | rows=1,000,000 | last_date=2014-04-07
Processed chunk 23 | rows=1,000,000 | last_date=2014-04-28
Processed chunk 24 | rows=1,000,000 | last_date=2014-05-18
Processed chunk 25 | rows=1,000,000 | last_date=2014-06-07
Processed chunk 26 | rows=1,000,000 | last_date=2014-06-26
Processed chunk 27 | rows=1,000,000 | last_date=2014-07-12
Processed chunk 28 | rows=1,000,000 | last_date=2014-07-27
Processed chunk 29 | rows=1,000,000 | last_date=2014-08-13
Processed chunk 30 | rows=1,000,000 | last_date=2014-09-01
Processed chunk 31 | rows=1,000,000 | last_date=2014-09-15
Processed chunk 32 | rows=1,000,000 | last_date=2014-09-29
Processed chunk 33 | rows=1,000,000 | last_date=2014-10-14
Processed chunk 34 | rows=1,000,000 | last_date=2014-10-28
Processed chunk 35 | rows=1,000,000 | last_date=2014-11-11
Processed chunk 36 | rows=1,000,000 | last_date=2014-11-25
Processed chunk 37 | rows=1,000,000 | last_date=2014-12-09
Processed chunk 38 | rows=1,000,000 | last_date=2014-12-

In [4]:
# Kiểm tra dung lượng file parquet sau khi chuyển đổi
size_mb = train_parquet.stat().st_size / 1024**2
print(f"Dung lượng train.parquet: {size_mb:.2f} MB")

Dung lượng train.parquet: 904.79 MB


In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

DATA_PROCESSED = Path("../data/processed")
DATA_INTERIM = Path("../data/interim")
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

parquet_file = DATA_PROCESSED / "train.parquet"

series_chunks = []
store_day_chunks = []

print("Đang quét train.parquet bằng iter_batches (sẽ mất khoảng 1-2 phút)...")
# Chỉ đọc các cột thật sự cần để tiết kiệm RAM
columns_to_read = ["date", "store_nbr", "item_nbr", "unit_sales"]

pf = pq.ParquetFile(parquet_file)
for batch in pf.iter_batches(columns=columns_to_read, batch_size=5_000_000):
    df = batch.to_pandas()
    # Kẹp âm theo đúng quy tắc D5
    df["unit_sales"] = df["unit_sales"].clip(lower=0)
    
    # Gom nhóm cục bộ cho từng chunk
    agg1 = df.groupby(["store_nbr", "item_nbr"]).agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        n_days=("date", "count")
    ).reset_index()
    series_chunks.append(agg1)
    
    agg2 = df.groupby(["date", "store_nbr"]).agg(
        n_rows=("item_nbr", "count"),
        sales_sum=("unit_sales", "sum")
    ).reset_index()
    store_day_chunks.append(agg2)

print("Đang tổng hợp dữ liệu (merge chunks)...")
# Tổng hợp toàn cục cho series profile
series_df = pd.concat(series_chunks, ignore_index=True)
series_df = series_df.groupby(["store_nbr", "item_nbr"]).agg(
    first_date=("first_date", "min"),
    last_date=("last_date", "max"),
    n_days=("n_days", "sum")
).reset_index()

# Tổng hợp toàn cục cho store-day profile
store_day_df = pd.concat(store_day_chunks, ignore_index=True)
store_day_df = store_day_df.groupby(["date", "store_nbr"]).agg(
    n_rows=("n_rows", "sum"),
    sales_sum=("sales_sum", "sum")
).reset_index()

print("Đang lưu CSV...")
series_df.to_csv(DATA_INTERIM / "train_series_profile.csv", index=False)
store_day_df.to_csv(DATA_INTERIM / "train_store_day.csv", index=False)

print(f"Hoàn tất!")
print(f"Số dòng series profile: {len(series_df):,}")
print(f"Số dòng store_day profile: {len(store_day_df):,}")


Đang quét train.parquet bằng iter_batches (sẽ mất khoảng 1-2 phút)...
Đang tổng hợp dữ liệu (merge chunks)...
Đang lưu CSV...
Hoàn tất!
Số dòng series profile: 174,685
Số dòng store_day profile: 83,606


In [2]:
import json

notebook_path = "01_data_checking.ipynb"
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

count = 0
for cell in nb.get("cells", []):
    if cell["cell_type"] in ["code", "markdown"]:
        new_source = []
        for line in cell.get("source", []):
            old_line = line
            # Chỉ thay thế chữ HOLDOUT viết hoa chữ V (để không đụng tới các biến code)
            line = line.replace("HOLDOUT", "HOLDOUT")
            if line != old_line:
                count += 1
            new_source.append(line)
        cell["source"] = new_source

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1, ensure_ascii=False)

print(f"Hoàn tất: Đã thay đổi {count} dòng chữ 'HOLDOUT' thành 'HOLDOUT' trong 01_data_checking.ipynb.")


Hoàn tất: Đã thay đổi 15 dòng chữ 'HOLDOUT' thành 'HOLDOUT' trong 01_data_checking.ipynb.
